In [1]:
from pathlib import Path
import subprocess
import sys
import re

In [9]:
# ===== 参数 =====
QUESTIONS_DIR = Path("Questions")
PYTHON_CMD = sys.executable
ENCODING = "utf-8"

USE_TIME_LIMITS = False          # 是否读取每题 limits.txt 中的时间限制
DEFAULT_TIMEOUT = 5.0           # USE_TIME_LIMITS=False 时使用
NORMALIZE_EOL = True           # 是否比较前统一换行
TRIM_END = True                # 是否比较前 rstrip
SAVE_CASE_OUTPUTS = False       # 是否把每个 code 对每个 input 的实际输出保存到文件
CASE_OUTPUTS_DIRNAME = "LLM Code Outputs"  # SAVE_CASE_OUTPUTS=True 时生效

def list_question_dirs(questions_dir: Path):
    return sorted(
        [p for p in questions_dir.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda x: x.name
    )

dirs = list_question_dirs(QUESTIONS_DIR)

In [3]:
def list_code_files(code_dir: Path):
    """
    识别文件名格式：
      {global_idx} {temperature}.txt
    例如：
      1 0.0.py
      12 0.2.py
    按 global_idx 排序返回 Path 列表
    """
    txts = []

    for p in code_dir.iterdir():
        if not (p.is_file() and p.suffix.lower() == ".py"):
            continue

        parts = p.stem.split()
        if len(parts) == 2 and parts[0].isdigit():
            txts.append((int(parts[0]), p))

    txts_sorted = sorted(txts, key=lambda x: x[0])
    return [p for _, p in txts_sorted]


def parse_idx_and_temp(txt_path: Path):
    parts = txt_path.stem.split()
    if len(parts) != 2 or not parts[0].isdigit():
        raise ValueError(f"非法文件名格式: {txt_path.name}")
    global_idx = int(parts[0])
    temperature = float(parts[1])
    return global_idx, temperature


def read_text(p: Path, encoding: str = ENCODING) -> str:
    return p.read_text(encoding=encoding, errors="replace")


def normalize_text(s: str, normalize_eol: bool = NORMALIZE_EOL, trim_end: bool = TRIM_END) -> str:
    if normalize_eol:
        s = s.replace("\r\n", "\n").replace("\r", "\n")
    if trim_end:
        s = s.rstrip()
    return s


def collect_cases_dir(in_dir: Path, out_dir: Path):
    if not in_dir.exists() or not in_dir.is_dir():
        raise ValueError(f"inputs 目录不存在: {in_dir}")
    if not out_dir.exists() or not out_dir.is_dir():
        raise ValueError(f"outputs 目录不存在: {out_dir}")

    in_files = sorted([p for p in in_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt"])
    out_map = {p.name: p for p in out_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt"}

    pairs = []
    for ip in in_files:
        op = out_map.get(ip.name)
        if op is not None:
            pairs.append((ip, op))

    if not pairs:
        raise ValueError(
            f"未找到匹配的 input/output 对。\n"
            f"Input dir: {in_dir}\n"
            f"Output dir: {out_dir}"
        )

    return pairs


def parse_time_limit_from_limits(limits_path: Path):
    """
    从 limits.txt 中提取时间限制，支持：
      Time: 1.00s
    """
    if not limits_path.exists():
        return None

    text = read_text(limits_path)
    m = re.search(r"Time:\s*([0-9]+(?:\.[0-9]+)?)\s*s", text, flags=re.IGNORECASE)
    if not m:
        return None
    return float(m.group(1))


def get_timeout_for_question(question_dir: Path):
    if not USE_TIME_LIMITS:
        return DEFAULT_TIMEOUT

    limits_path = question_dir / "limits.txt"
    parsed = parse_time_limit_from_limits(limits_path)
    if parsed is None:
        return DEFAULT_TIMEOUT
    return parsed


def run_python_solution(solution_path: Path, inp: str, timeout: float):
    """
    运行单个 .py 解答，返回：
      returncode, stdout, stderr, timeout_flag
    """
    try:
        res = subprocess.run(
            [PYTHON_CMD, "-u", solution_path.name],
            input=inp,
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=str(solution_path.parent) if solution_path.parent.exists() else None,
        )
        return res.returncode, res.stdout, res.stderr, False
    except subprocess.TimeoutExpired as e:
        out = e.stdout if isinstance(e.stdout, str) and e.stdout is not None else ""
        err = e.stderr if isinstance(e.stderr, str) and e.stderr is not None else ""
        return -1, out, err, True


def judge_one_code(code_path: Path, case_pairs, timeout: float, save_case_outputs: bool = SAVE_CASE_OUTPUTS, output_root: Path | None = None):
    """
    对单个 code 跑完全部 input/output，返回：
    {
        "global_idx": int,
        "temperature": float,
        "ac_num": int,
        "total_num": int,
        "details": [...]
    }
    """
    global_idx, temperature = parse_idx_and_temp(code_path)
    ac_num = 0
    total_num = len(case_pairs)
    details = []

    code_stem = code_path.stem

    if save_case_outputs and output_root is not None:
        code_output_dir = output_root / code_stem
        code_output_dir.mkdir(parents=True, exist_ok=True)
    else:
        code_output_dir = None

    for in_path, out_path in case_pairs:
        inp = read_text(in_path)
        expected = read_text(out_path)

        rc, got, err, to = run_python_solution(code_path, inp, timeout=timeout)

        if code_output_dir is not None:
            save_path = code_output_dir / out_path.name
            save_path.write_text(got, encoding=ENCODING)

        got_cmp = normalize_text(got)
        exp_cmp = normalize_text(expected)

        if to:
            status = "TLE"
        elif rc != 0:
            status = "RE"
        elif got_cmp == exp_cmp:
            status = "AC"
            ac_num += 1
        else:
            status = "WA"

        details.append({
            "case": in_path.name,
            "status": status,
            "returncode": rc,
            "stderr": err,
        })

    return {
        "global_idx": global_idx,
        "temperature": temperature,
        "ac_num": ac_num,
        "total_num": total_num,
        "details": details,
    }


def judge_one_question(question_dir: Path):
    """
    对单个题目目录：
    - 读取 inputs / outputs
    - 遍历 LLM Codes 中所有 code
    - 生成 score.txt
    - 追加三类统计：
        1) 全 AC 的 code 数量 / 总 code 样本量
        2) 部分 AC 的 code 数量 / 总 code 样本量
        3) 完全 WA 的 code 数量 / 总 code 样本量
    返回：
      list[dict]
    """
    inputs_dir = question_dir / "inputs"
    outputs_dir = question_dir / "outputs"
    code_dir = question_dir / "LLM Codes"
    score_path = question_dir / "score.txt"

    if not code_dir.exists() or not code_dir.is_dir():
        raise FileNotFoundError(f"{question_dir} 下不存在 LLM Codes 文件夹")

    case_pairs = collect_cases_dir(inputs_dir, outputs_dir)
    code_files = list_code_files(code_dir)
    timeout = get_timeout_for_question(question_dir)

    if SAVE_CASE_OUTPUTS:
        output_root = question_dir / CASE_OUTPUTS_DIRNAME
        output_root.mkdir(parents=True, exist_ok=True)
    else:
        output_root = None

    results = []

    print(f"{question_dir.name}")
    for code_path in code_files:
        result = judge_one_code(
            code_path=code_path,
            case_pairs=case_pairs,
            timeout=timeout,
            save_case_outputs=SAVE_CASE_OUTPUTS,
            output_root=output_root,
        )
        results.append(result)
        print(f"done {code_path.name} => {result['ac_num']}/{result['total_num']}")

    # 逐个 code 的明细
    lines = [
        f"{r['global_idx']}: {r['ac_num']}/{r['total_num']}"
        for r in sorted(results, key=lambda x: x["global_idx"])
    ]

    # 新增：每道题三类统计
    total_codes = len(results)
    fully_ac_codes = sum(1 for r in results if r["ac_num"] == r["total_num"])
    partial_ac_codes = sum(1 for r in results if 0 < r["ac_num"] < r["total_num"])
    fully_wa_codes = sum(1 for r in results if r["ac_num"] == 0)

    lines.extend([
        "",
        f"All AC codes: {fully_ac_codes}/{total_codes}",
        f"Partially AC codes: {partial_ac_codes}/{total_codes}",
        f"Completely WA codes: {fully_wa_codes}/{total_codes}",
    ])

    score_path.write_text("\n".join(lines), encoding=ENCODING)

    print(f"score written: {score_path}")
    print(
        f"{question_dir.name} summary -> "
        f"All AC: {fully_ac_codes}/{total_codes}, "
        f"Partially AC: {partial_ac_codes}/{total_codes}, "
        f"Completely WA: {fully_wa_codes}/{total_codes}"
    )
    return results

In [4]:
display(dirs)

[PosixPath('Questions/Easy B3666'),
 PosixPath('Questions/Easy P15288'),
 PosixPath('Questions/Easy P15457'),
 PosixPath('Questions/Easy P4306'),
 PosixPath('Questions/Easy P7714'),
 PosixPath('Questions/Hard P11658'),
 PosixPath('Questions/Hard P11823'),
 PosixPath('Questions/Hard P13901'),
 PosixPath('Questions/Hard P15082'),
 PosixPath('Questions/Hard P6845'),
 PosixPath('Questions/ML Q1'),
 PosixPath('Questions/ML Q2'),
 PosixPath('Questions/ML Q3'),
 PosixPath('Questions/Mid P1407'),
 PosixPath('Questions/Mid P14989'),
 PosixPath('Questions/Mid P3007'),
 PosixPath('Questions/Mid P3167'),
 PosixPath('Questions/Mid P4092')]

In [ ]:
# 循环处理全部题目
all_judge_results = {}

for d in dirs:
    try:
        all_judge_results[d.name] = judge_one_question(d)
    except Exception as e:
        print(f"error: {d.name} -> {e}")

In [12]:
# 处理index0版本
d = dirs[4]

try:
    test_judge_results = judge_one_question(d)
except Exception as e:
    print(f"error: {d.name} -> {e}")

Easy P15457
done 1 0.0.py => 9/9
done 2 0.0.py => 9/9
done 3 0.0.py => 9/9
done 4 0.0.py => 9/9
done 5 0.0.py => 9/9
done 6 0.0.py => 9/9
done 7 0.0.py => 9/9
done 8 0.0.py => 9/9
done 9 0.0.py => 9/9
done 10 0.0.py => 9/9
done 11 0.0.py => 9/9
done 12 0.0.py => 9/9
done 13 0.0.py => 9/9
done 14 0.0.py => 9/9
done 15 0.0.py => 9/9
done 16 0.0.py => 9/9
done 17 0.0.py => 9/9
done 18 0.0.py => 9/9
done 19 0.0.py => 9/9
done 20 0.0.py => 9/9
done 21 0.2.py => 4/9
done 22 0.2.py => 9/9
done 23 0.2.py => 9/9
done 24 0.2.py => 0/9
done 25 0.2.py => 8/9
done 26 0.2.py => 9/9
done 27 0.2.py => 9/9
done 28 0.2.py => 9/9
done 29 0.2.py => 9/9
done 30 0.2.py => 0/9
done 31 0.2.py => 9/9
done 32 0.2.py => 9/9
done 33 0.2.py => 7/9
done 34 0.2.py => 9/9
done 35 0.2.py => 9/9
done 36 0.2.py => 9/9
done 37 0.2.py => 0/9
done 38 0.2.py => 9/9
done 39 0.2.py => 9/9
done 40 0.2.py => 9/9
done 41 0.5.py => 9/9
done 42 0.5.py => 9/9
done 43 0.5.py => 9/9
done 44 0.5.py => 9/9
done 45 0.5.py => 9/9
done 46

In [19]:
run_python_solution(solution_path=Path("./Questions/Easy B3666/LLM Codes/1 0.0.py"), inp="5\n2 1 3 5 4", timeout=10.0)

(0, '1\n3\n3\n4\n1', '', False)